# gld.Controlo_Sanitario

In [ ]:
#Parameter

run_id = ""

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

print(f"{'='*80}")
print("CONSTRUÇÃO DA TABELA DE FACTOS: gld.fact_controlo_sanitario")
print(f"{'='*80}\n")


# 1. CARREGAR TABELAS (SILVER E DIMENSÕES GOLD)

df_slv_tri = spark.read.table("slv.trichinella")
df_slv_rep = spark.read.table("slv.motivosreprovacao")
df_dim_exp = spark.read.table("gld.dim_exploracao_scd")
df_dim_mat = spark.read.table("gld.dim_matadouro")
# Novas tabelas carregadas
df_dim_mat_geo = spark.read.table("gld.dim_mat_geografia")
df_slv_cotacoes = spark.read.table("slv.cotacoes_carne")


# 2. PREPARAÇÃO E AGREGAÇÃO: ABATES (TRICHINELLA)
df_fact = df_slv_tri.withColumnRenamed("data_colheita", "data") \
                    .withColumn("ncv", F.trim(F.regexp_replace(F.col("ncv"), r"(?i)\s*\(CANCELADO\)", "")))

df_agg_tri = df_fact.groupBy("data", "marca", "ncv").agg(
    F.sum("total_testados").alias("abatidos"),
    F.first("meta_source_file").alias("meta_source_file"),
    F.max("meta_source_file_date").alias("meta_source_file_date")
)


# 3. PREPARAÇÃO E AGREGAÇÃO: REPROVAÇÕES
df_rep_prep = df_slv_rep.withColumnRenamed("data_controlo", "data") \
                        .withColumn("ncv", F.trim(F.regexp_replace(F.col("ncv"), r"(?i)\s*\(CANCELADO\)", "")))

df_agg_rep = df_rep_prep.groupBy("data", "marca", "ncv").agg(
    F.sum("qtd_animais_reprovados").alias("reprovados")
)


# 4. CONSOLIDAÇÃO DA FACT TABLE & REGRAS DE NEGÓCIO
df_base_fact = df_agg_tri.join(df_agg_rep, ["data", "marca", "ncv"], "left")
df_base_fact = df_base_fact.withColumn("reprovados", F.coalesce(F.col("reprovados"), F.lit(0)))

linhas_inconsistentes = df_base_fact.filter(F.col("reprovados") > F.col("abatidos")).count()
df_base_fact = df_base_fact.filter(F.col("reprovados") <= F.col("abatidos"))


# 5. JOIN 1: DIMENSÃO EXPLORAÇÃO (SCD TIPO 2)
condicao_exp = [
    df_base_fact.marca == df_dim_exp.marca,
    df_base_fact.data >= df_dim_exp.data_inicio,
    df_base_fact.data <= df_dim_exp.data_fim
]

df_join1 = df_base_fact.join(df_dim_exp, condicao_exp, "left") \
    .select(
        df_base_fact["*"],
        df_dim_exp["SK_Marca_Historico"].alias("SK_Marca_Historico"),
        df_dim_exp["SK_Geo_Exp"]
    )


# 6. JOIN 2: DIMENSÃO MATADOURO
condicao_mat = [df_join1.ncv == df_dim_mat.ncv]

df_join2 = df_join1.join(df_dim_mat, condicao_mat, "left") \
    .select(
        df_join1["*"],
        df_dim_mat["SK_Matadouro"],
        df_dim_mat["SK_Geo_Mat"]
    )


# 7. JOIN 3: DIMENSÃO GEOGRAFIA DO MATADOURO (Recuperar o DSAVR)
condicao_mat_geo = [df_join2.SK_Geo_Mat == df_dim_mat_geo.SK_Geo_Mat]

df_join3 = df_join2.join(df_dim_mat_geo, condicao_mat_geo, "left") \
    .select(
        df_join2["*"],
        df_dim_mat_geo["dsavr"].alias("dsavr_temp") # Coluna temporária para o Join 4
    )


# 8. JOIN 4: DIMENSÃO COTAÇÕES DE MERCADO (SCD TIPO 2)
condicao_cotacoes = [
    df_join3.dsavr_temp == df_slv_cotacoes.dsavr,
    df_join3.data >= df_slv_cotacoes.data_inicio_historico,
    df_join3.data <= df_slv_cotacoes.data_fim_historico
]

df_join4 = df_join3.join(df_slv_cotacoes, condicao_cotacoes, "left") \
    .select(
        df_join3["*"],
        df_slv_cotacoes["preco_kg"]
    )


# 9. CALCULO COLUNAS CALCULADAS
df_calculado = df_join4.withColumn("valor_estimado_por_suino", F.col("preco_kg") * 80) \
                             .withColumn("valor_por_animais_abatidos", F.col("abatidos") * F.col("valor_estimado_por_suino")) \
                             .withColumn("valor_por_animais_reprovados", F.col("reprovados") * F.col("valor_estimado_por_suino"))


# 10. SELEÇÃO FINAL E AUDITORIA
df_final = df_calculado.select(
    # --- Chaves Estrangeiras (Foreign Keys) ---
    "SK_Marca_Historico",
    "SK_Geo_Exp",
    "SK_Matadouro",
    "SK_Geo_Mat",
    
    # --- Colunas de Negócio (Factos) ---
    "data",
    "abatidos",
    "reprovados", 

    # --- Colunas Calculadas:
    "valor_estimado_por_suino",
    "valor_por_animais_abatidos",
    "valor_por_animais_reprovados",


    # --- Colunas Temporárias (Para validação) ---
    F.col("marca").alias("marca_TO_DROP"),
    F.col("ncv").alias("ncv_TO_DROP"),
    F.col("dsavr_temp").alias("dsavr_TO_DROP"), # Ajuda a validar falhas nas cotações
    
    # --- Auditoria e Metadados ---
    "meta_source_file",
    "meta_source_file_date",
    F.lit("slv.trichinella | slv.motivosreprovacao").alias("audit_source_silver"),
    F.current_timestamp().alias("audit_gold_refresh_timestamp")
)


# 11. Criar chave técnica na fact
df_final = df_final.withColumn(
    "SK_Fact_Controlo_Sanitario",
    F.concat(
        F.lit("SK_FACT_CONTROLO_"),
        F.md5(
            F.concat_ws(
                "|",
                F.coalesce(F.col("data").cast("string"), F.lit("N/A")),
                F.coalesce(F.upper(F.trim(F.col("marca_TO_DROP"))), F.lit("N/A")),
                F.coalesce(F.upper(F.trim(F.col("ncv_TO_DROP"))), F.lit("N/A"))
            )
        )
    )
)

#Deduplicar source antes do merge, escolhendo a linha com menos nulls (em caso de empate escolhe a mais recente)
cols_para_avaliar_nulls = [
    "SK_Marca_Historico",
    "SK_Geo_Exp",
    "SK_Matadouro",
    "SK_Geo_Mat",
    "valor_estimado_por_suino",
    "valor_por_animais_abatidos",
    "valor_por_animais_reprovados",
    "dsavr_TO_DROP"
]

df_final = df_final.withColumn(
    "qtd_nulls_linha",
    sum([
        F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0))
        for c in cols_para_avaliar_nulls
    ])
)

window_fact = Window.partitionBy("SK_Fact_Controlo_Sanitario") \
    .orderBy(
        F.col("qtd_nulls_linha").asc(),
        F.col("meta_source_file_date").desc_nulls_last()
    )

df_final = df_final.withColumn("rn_fact", F.row_number().over(window_fact)) \
                   .filter(F.col("rn_fact") == 1) \
                   .drop("rn_fact", "qtd_nulls_linha")


# 12. UPSERT / MERGE NA GOLD

spark.sql("CREATE SCHEMA IF NOT EXISTS gld")

target_table = "gld.fact_controlo_sanitario"

if not spark.catalog.tableExists(target_table):

    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(
        f"Tabela {target_table} criada com "
        f"{df_final.count()} registos."
    )

else:

    delta_target = DeltaTable.forName(
        spark,
        target_table
    )

    merge_condition = """
        target.SK_Fact_Controlo_Sanitario =
        source.SK_Fact_Controlo_Sanitario
    """

    # Comparar todas as colunas, exceto:
    # - a chave usada no MERGE;
    # - o timestamp, que muda em todas as execuções.
    colunas_comparacao = [
        coluna
        for coluna in df_final.columns
        if coluna not in [
            "SK_Fact_Controlo_Sanitario",
            "audit_gold_refresh_timestamp"
        ]
    ]

    # Atualizar apenas quando algum valor de negócio mudou.
    # <=> é uma comparação null-safe.
    condicao_mudanca = " OR ".join([
        f"NOT (target.`{coluna}` <=> source.`{coluna}`)"
        for coluna in colunas_comparacao
    ])

    (
        delta_target.alias("target")
        .merge(
            df_final.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll(
            condition=condicao_mudanca
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merge concluído na tabela {target_table}."
    )


print(
    f"Sucesso! DataFrame processado com "
    f"{df_final.count()} registos."
)

print(
    f"ATENÇÃO: {linhas_inconsistentes} linhas foram removidas "
    "porque 'reprovados > abatidos'."
)



StatementMeta(, 958c6549-548e-4d8f-90be-8dc2b42b5ad4, 10, Finished, Available, Finished, False)

CONSTRUÇÃO DA TABELA DE FACTOS: gld.fact_controlo_sanitario



Tabela gld.fact_controlo_sanitario criada com 507309 registos.


Sucesso! Tabela de factos gerada com 507309 registos.
ATENÇÃO: 6 linhas foram removidas porque 'reprovados > abatidos'.


# gld.Reprovaçoes

In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

print(f"{'='*80}")
print("CONSTRUÇÃO DA TABELA DE FACTOS: gld.fact_reprovacoes")
print(f"{'='*80}\n")


# 1. CARREGAR TABELAS (SILVER E DIMENSÕES GOLD)

df_slv_rep = spark.read.table("slv.motivosreprovacao")

df_dim_exp = spark.read.table("gld.dim_exploracao_scd") 
df_dim_mat = spark.read.table("gld.dim_matadouro")
df_dim_mot = spark.read.table("gld.dim_motivorejeicao")

# Novas tabelas para o caminho até às cotações
df_dim_mat_geo = spark.read.table("gld.dim_mat_geografia")
df_slv_cotacoes = spark.read.table("slv.cotacoes_carne")


# 2. LIMPEZA E AGREGAÇÃO DA TABELA DE FACTOS (REPROVAÇÕES)

# Renomear data e limpar NCV com a mesma regra Case Insensitive (?i)
df_fact = df_slv_rep.withColumnRenamed("data_controlo", "data") \
                    .withColumn("ncv", F.trim(F.regexp_replace(F.col("ncv"), r"(?i)\s*\(CANCELADO\)", "")))

# Agregação na granularidade exigida (ncv, data, marca, motivo_rejeicao)
df_agg = df_fact.groupBy("ncv", "data", "marca", "motivo_rejeicao").agg(
    F.sum("qtd_animais_reprovados").alias("reprovados"),
    # Metadados de rastreabilidade
    F.first("meta_source_file").alias("meta_source_file"),
    F.max("meta_source_file_date").alias("meta_source_file_date")
)


# 3. JOIN 1: DIMENSÃO EXPLORAÇÃO (SCD TIPO 2)

condicao_exp = [
    df_agg.marca == df_dim_exp.marca,
    df_agg.data >= df_dim_exp.data_inicio,
    df_agg.data <= df_dim_exp.data_fim
]

df_join1 = df_agg.join(df_dim_exp, condicao_exp, "left") \
    .select(
        df_agg["*"],
        df_dim_exp["SK_Marca_Historico"].alias("SK_Marca_Historico"),
        df_dim_exp["SK_Geo_Exp"]
    )


# 4. JOIN 2: DIMENSÃO MATADOURO

condicao_mat = [df_join1.ncv == df_dim_mat.ncv]

df_join2 = df_join1.join(df_dim_mat, condicao_mat, "left") \
    .select(
        df_join1["*"],
        df_dim_mat["SK_Matadouro"],
        df_dim_mat["SK_Geo_Mat"]
    )


# 5. JOIN 3: DIMENSÃO MOTIVOS DE REPROVAÇÃO
condicao_mot = [df_join2.motivo_rejeicao == df_dim_mot.motivo_rejeicao]

df_join3 = df_join2.join(df_dim_mot, condicao_mot, "left") \
    .select(
        df_join2["*"],
        df_dim_mot["SK_Motivo_Rejeicao"]
    )


# 6. JOIN 4: DIMENSÃO GEOGRAFIA DO MATADOURO (Recuperar o DSAVR)
condicao_mat_geo = [df_join3.SK_Geo_Mat == df_dim_mat_geo.SK_Geo_Mat]

df_join4 = df_join3.join(df_dim_mat_geo, condicao_mat_geo, "left") \
    .select(
        df_join3["*"],
        df_dim_mat_geo["dsavr"].alias("dsavr_temp")
    )


# 7. JOIN 5: DIMENSÃO COTAÇÕES DE MERCADO (SCD TIPO 2) 
condicao_cotacoes = [
    df_join4.dsavr_temp == df_slv_cotacoes.dsavr,
    df_join4.data >= df_slv_cotacoes.data_inicio_historico,
    df_join4.data <= df_slv_cotacoes.data_fim_historico
]

df_join5 = df_join4.join(df_slv_cotacoes, condicao_cotacoes, "left") \
    .select(
        df_join4["*"],
        df_slv_cotacoes["preco_kg"]
    )


# 8. CALCULO COLUNAS CALCULADAS

df_calculado = df_join5.withColumn("valor_estimado_por_suino", F.col("preco_kg") * 80) \
.withColumn("valor_por_animais_reprovados", F.col("reprovados") * F.col("valor_estimado_por_suino"))


# 9. SELEÇÃO FINAL E AUDITORIA

df_final = df_calculado.select(
    # --- Chaves Estrangeiras (Foreign Keys) ---
    "SK_Marca_Historico",
    "SK_Geo_Exp",
    "SK_Matadouro",
    "SK_Geo_Mat",
    "SK_Motivo_Rejeicao",
    
    # --- Colunas de Negócio (Factos) ---
    "data",
    "reprovados",

    # --- Colunas Calculadas:
    "valor_estimado_por_suino",
    "valor_por_animais_reprovados",

    # --- Colunas Temporárias (Marcadas para Remover no Futuro) ---
    F.col("marca").alias("marca_TO_DROP"),
    F.col("ncv").alias("ncv_TO_DROP"),
    F.col("motivo_rejeicao").alias("motivo_TO_DROP"),
    F.col("dsavr_temp").alias("dsavr_TO_DROP"), # Útil para auditoria do pricing
    
    # --- Auditoria e Metadados ---
    "meta_source_file",
    "meta_source_file_date",
    F.lit("slv.motivosreprovacao").alias("audit_source_silver"),
    F.current_timestamp().alias("audit_gold_refresh_timestamp")
)

# 10. CRIAR CHAVE TÉCNICA DA FACT REPROVAÇÕES

df_final = df_final.withColumn(
    "SK_Fact_Reprovacoes",
    F.concat(
        F.lit("SK_FACT_REPROVACOES_"),
        F.md5(
            F.concat_ws(
                "|",
                F.coalesce(F.col("data").cast("string"), F.lit("N/A")),
                F.coalesce(F.upper(F.trim(F.col("marca_TO_DROP"))), F.lit("N/A")),
                F.coalesce(F.upper(F.trim(F.col("ncv_TO_DROP"))), F.lit("N/A")),
                F.coalesce(F.upper(F.trim(F.col("motivo_TO_DROP"))), F.lit("N/A"))
            )
        )
    )
)

# 11. DEDUPLICAR SOURCE ANTES DO MERGE
# Escolhe a linha com menos nulls; em empate, a mais recente

cols_para_avaliar_nulls = [
    "SK_Marca_Historico",
    "SK_Geo_Exp",
    "SK_Matadouro",
    "SK_Geo_Mat",
    "SK_Motivo_Rejeicao",
    "valor_estimado_por_suino",
    "valor_por_animais_reprovados",
    "dsavr_TO_DROP"
]

df_final = df_final.withColumn(
    "qtd_nulls_linha",
    sum([
        F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0))
        for c in cols_para_avaliar_nulls
    ])
)

window_fact = Window.partitionBy("SK_Fact_Reprovacoes") \
    .orderBy(
        F.col("qtd_nulls_linha").asc(),
        F.col("meta_source_file_date").desc_nulls_last()
    )

df_final = df_final.withColumn("rn_fact", F.row_number().over(window_fact)) \
                   .filter(F.col("rn_fact") == 1) \
                   .drop("rn_fact", "qtd_nulls_linha")


# 12. UPSERT / MERGE NA GOLD

spark.sql("CREATE SCHEMA IF NOT EXISTS gld")

target_table = "gld.fact_reprovacoes"

if not spark.catalog.tableExists(target_table):

    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(
        f"Tabela {target_table} criada com "
        f"{df_final.count()} registos."
    )

else:

    delta_target = DeltaTable.forName(
        spark,
        target_table
    )

    colunas_comparacao = [
        coluna
        for coluna in df_final.columns
        if coluna not in [
            "SK_Fact_Reprovacoes",
            "audit_gold_refresh_timestamp"
        ]
    ]

    condicao_mudanca = " OR ".join([
        f"NOT (target.`{coluna}` <=> source.`{coluna}`)"
        for coluna in colunas_comparacao
    ])

    (
        delta_target.alias("target")
        .merge(
            df_final.alias("source"),
            """
            target.SK_Fact_Reprovacoes =
            source.SK_Fact_Reprovacoes
            """
        )
        .whenMatchedUpdateAll(
            condition=condicao_mudanca
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merge concluído na tabela {target_table}."
    )


total_gold = spark.table(target_table).count()

print(
    f"Sucesso! Tabela {target_table} contém "
    f"{total_gold} registos."
)



StatementMeta(, 4f4b1534-b075-4d08-81dc-c858a275aa4d, 3, Finished, Available, Finished, False)

CONSTRUÇÃO DA TABELA DE FACTOS: gld.fact_reprovacoes

oi
Merge concluído na tabela gld.fact_reprovacoes.
Sucesso! Tabela de factos gerada com 39692 registos.


## Criação de tabela de auditoria

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql import functions as F



# VALIDAR RUN ID RECEBIDO DO PIPELINE

if run_id is None or str(run_id).strip() == "":
    raise ValueError(
        "O parâmetro 'run_id' não foi recebido do pipeline. "
        "Confirma se a atividade Notebook está configurada com "
        "run_id = @pipeline().parameters.p_run_id"
    )

run_id = str(run_id).strip()

print(f"Run ID recebido do pipeline: {run_id}")



# CALCULAR MÉTRICAS REAIS ENTRE DUAS VERSÕES DELTA

def calcular_metricas_delta(
    table_name,
    chaves,
    colunas_ignorar=None
):
    colunas_ignorar = colunas_ignorar or []

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"A tabela '{table_name}' não existe."
        )

    delta_table = DeltaTable.forName(
        spark,
        table_name
    )

    # São necessárias apenas as duas versões mais recentes
    historico = (
        delta_table.history(2)
        .select(
            "version",
            "timestamp",
            "operation"
        )
        .orderBy(
            F.col("version").desc()
        )
    )

    versoes = [
        row["version"]
        for row in historico
        .select("version")
        .collect()
    ]

    if not versoes:
        raise ValueError(
            f"Não existe histórico Delta para '{table_name}'."
        )

    versao_atual = versoes[0]

    df_atual = (
        spark.read
        .format("delta")
        .option(
            "versionAsOf",
            versao_atual
        )
        .table(table_name)
    )

    total_rows = df_atual.count()

    # Primeira versão da tabela
    if len(versoes) == 1:
        return {
            "current_version": versao_atual,
            "previous_version": None,
            "total_rows": total_rows,
            "rows_added": total_rows,
            "rows_updated": 0,
            "rows_deleted": 0
        }

    versao_anterior = versoes[1]

    df_anterior = (
        spark.read
        .format("delta")
        .option(
            "versionAsOf",
            versao_anterior
        )
        .table(table_name)
    )

    # Validar as chaves
    for chave in chaves:
        if chave not in df_atual.columns:
            raise ValueError(
                f"A chave '{chave}' não existe na versão atual "
                f"da tabela '{table_name}'. "
                f"Colunas disponíveis: {df_atual.columns}"
            )

        if chave not in df_anterior.columns:
            raise ValueError(
                f"A chave '{chave}' não existe na versão anterior "
                f"da tabela '{table_name}'."
            )

    chaves_atuais = (
        df_atual
        .select(*chaves)
        .distinct()
    )

    chaves_anteriores = (
        df_anterior
        .select(*chaves)
        .distinct()
    )

    # Linhas adicionadas
    rows_added = (
        chaves_atuais
        .join(
            chaves_anteriores,
            on=chaves,
            how="left_anti"
        )
        .count()
    )

    # Linhas eliminadas
    rows_deleted = (
        chaves_anteriores
        .join(
            chaves_atuais,
            on=chaves,
            how="left_anti"
        )
        .count()
    )

    # Colunas de negócio a comparar
    colunas_comparacao = [
        coluna
        for coluna in df_atual.columns
        if coluna in df_anterior.columns
        and coluna not in chaves
        and coluna not in colunas_ignorar
    ]

    atual = df_atual.alias("atual")
    anterior = df_anterior.alias("anterior")

    condicao_join = F.lit(True)

    for chave in chaves:
        condicao_join = (
            condicao_join
            & F.col(f"atual.{chave}")
            .eqNullSafe(
                F.col(f"anterior.{chave}")
            )
        )

    condicao_alteracao = F.lit(False)

    for coluna in colunas_comparacao:
        condicao_alteracao = (
            condicao_alteracao
            | ~F.col(f"atual.{coluna}")
            .eqNullSafe(
                F.col(f"anterior.{coluna}")
            )
        )

    rows_updated = (
        atual
        .join(
            anterior,
            on=condicao_join,
            how="inner"
        )
        .filter(condicao_alteracao)
        .select(
            *[
                F.col(f"atual.{chave}").alias(chave)
                for chave in chaves
            ]
        )
        .distinct()
        .count()
    )

    return {
        "current_version": versao_atual,
        "previous_version": versao_anterior,
        "total_rows": total_rows,
        "rows_added": rows_added,
        "rows_updated": rows_updated,
        "rows_deleted": rows_deleted
    }



# ESCREVER AUDITORIA EM APPEND

def escrever_auditoria_pipeline(
    layer,
    notebook_name,
    table_name,
    metricas,
    run_id,
    status="success"
):
    if run_id is None or str(run_id).strip() == "":
        raise ValueError(
            f"Não é possível auditar '{table_name}': "
            "run_id está vazio."
        )

    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS audit"
    )

    audit_df = (
        spark.createDataFrame([
            Row(
                run_id=str(run_id).strip(),
                layer=str(layer),
                notebook_name=str(notebook_name),
                table_name=str(table_name),
                status=str(status),
                total_rows=int(metricas["total_rows"]),
                rows_added=int(metricas["rows_added"]),
                rows_updated=int(metricas["rows_updated"]),
                rows_deleted=int(metricas["rows_deleted"])
            )
        ])
        .withColumn(
            "audit_timestamp",
            F.current_timestamp()
        )
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(
            "audit.pipeline_audit_log"
        )
    )

    print(
        f"Auditoria registada | "
        f"run_id={run_id} | "
        f"table={table_name} | "
        f"versão anterior={metricas['previous_version']} | "
        f"versão atual={metricas['current_version']} | "
        f"total={metricas['total_rows']} | "
        f"added={metricas['rows_added']} | "
        f"updated={metricas['rows_updated']} | "
        f"deleted={metricas['rows_deleted']}"
    )



# CONFIGURAÇÃO DAS FACTS

tabela_controlo = "gld.fact_controlo_sanitario"
tabela_reprovacoes = "gld.fact_reprovacoes"

chaves_controlo = [
    "SK_Fact_Controlo_Sanitario"
]

chaves_reprovacoes = [
    "SK_Fact_Reprovacoes"
]



# COLUNAS QUE NÃO DEVEM CONTAR COMO ALTERAÇÃO
colunas_ignorar = [
    "audit_gold_refresh_timestamp"
]



# CALCULAR MÉTRICAS DAS FACTS
metricas_controlo = calcular_metricas_delta(
    table_name=tabela_controlo,
    chaves=chaves_controlo,
    colunas_ignorar=colunas_ignorar
)

metricas_reprovacoes = calcular_metricas_delta(
    table_name=tabela_reprovacoes,
    chaves=chaves_reprovacoes,
    colunas_ignorar=colunas_ignorar
)

print("Métricas fact_controlo_sanitario:")
print(metricas_controlo)

print("Métricas fact_reprovacoes:")
print(metricas_reprovacoes)



# ESCREVER NA AUDITORIA COM O RUN ID DO PIPELINE
escrever_auditoria_pipeline(
    layer="gold",
    notebook_name="fact_controlo_sanitario",
    table_name=tabela_controlo,
    metricas=metricas_controlo,
    run_id=run_id
)

escrever_auditoria_pipeline(
    layer="gold",
    notebook_name="fact_reprovacoes",
    table_name=tabela_reprovacoes,
    metricas=metricas_reprovacoes,
    run_id=run_id
)

print(
    f"Auditoria das facts concluída com sucesso. "
    f"Run ID: {run_id}"
)

## Validações

In [ ]:
# from pyspark.sql import functions as F


# # LER FACT TABLE

# df_fact = spark.read.table("gld.fact_controlo_sanitario")


# # TOTAL DE LINHAS
# total_linhas = df_fact.count()

# print(f"\nTOTAL DE LINHAS NA FACT: {total_linhas}")


# # TOTAL DE NULLS POR COLUNA
# print("\nNULLS POR COLUNA:")

# df_nulls = df_fact.select([
#     F.count(
#         F.when(F.col(c).isNull(), c)
#     ).alias(c)
#     for c in df_fact.columns
# ])

# display(df_nulls)


# # TOTAL GLOBAL DE NULLS NA TABELA
# total_nulls_expr = []

# for c in df_fact.columns:
#     total_nulls_expr.append(
#         F.count(F.when(F.col(c).isNull(), c))
#     )

# total_nulls = df_fact.select(total_nulls_expr) \
#     .collect()[0]

# total_nulls_global = sum(total_nulls)

# print(f"\nTOTAL GLOBAL DE NULLS NA FACT: {total_nulls_global}")

StatementMeta(, 4fbe7486-9800-4d2c-b9a9-567c5ba84abb, 4, Finished, Available, Finished, False)


TOTAL DE LINHAS NA FACT: 507312

NULLS POR COLUNA:


SynapseWidget(Synapse.DataFrame, 97a4877a-0404-4582-a1fd-c5720594860d)


TOTAL GLOBAL DE NULLS NA FACT: 640697
